# Denoising Diffusion Probabilistic Models for Turbulence


## Overview

Generative methods are a class of machine learning models which learn to draw samples from a theoretical underlying data distrbution which encapsulates the training data, typically denoted as $p_{data}$. This can be anything from drawing samples of text which have a high probability of aligning to an input text prompt (such as ChatGPT!), to creating images and videos based on arbitrary provided information (conditional generation), or generating a plausible output from nothing but noise. Generative methods present a departure from the traditional field of Machine Learning, as they are no longer advanced interpolation mechanisms, rigidly held to the boundaries of the training data, and provided sufficient data, allow for a statistical representation of input data. Concretely, trained on turbulence data, a trained generative model allows for exploration of plausible, realisable flow states, such as what may be achieved by varying the random seed of an initial flow state. 

This tutorial focuses on a specific generative model, which is widely used across several domains today -- the diffusion model. An overview of the Denoising Diffusion Probabilistic Model (DDPM) generative method is provided, a powerful tool for generating synthetic turbulence. **Unconditional generation** is dealt with in depth, and extensions to condtitional generation are considered.
An introduction is provided to the method, applications to conditional generation -- such as super-resolution, and video generation -- as well as unconditional generation. **Prerequisites: Convolutional Neural Networks, Evidence Lower Bound (ELBO)**. 

## 1. Introduction to DDPM

DDPMs (<cite>[Ho et al., 2020][1]</cite>) are a specific formulation of the general Diffusion Model (<cite>[Song and Ermon, 2021][2]</cite>), which begins from the result that under certain conditions, forward-in-time diffusion processes have a tractable reverse-in-time counterpart. In the context of DDPM, this forward process is an additive noise operator based on a Gaussian transition kernel. 

Effectively, a maximum and minimum additive noise variance is prescribed, the distance between which is discretised into a number of steps. These steps define a Markov transition process from the original data distribution, to a unit Gaussian distribution. Samples from the dataset are noised to varying degrees, and a noise-predicting neural network is trained to predict the noise added to the original steps. The reverse process can then be computed through reverse-in-time transitions, using the noise-prediction neural network to iteratively denoise from Gaussian noise to a sample of data. 

### 1.1 Standard Formulation

In this subsection, the original formulation of DDPM is descibed. 

---

#### **Forward process**

The parameter $\beta$ is usually used to refer to the variance of the Gaussian noise added at each step, with its counterpart $\alpha=1-\beta$, and $\bar{\alpha}$ used to refer to $\Pi \alpha$ over all noising timesteps. The noising process is defined in terms of transition kernels, and is denoted with the letter $q$. Transition kernels take the form of a Gaussian distribution, with the mean and variance scaled in accordance to the previous state. Essentially, at $t=0$, there is no noise added to a sample of data. At $t=T$, where $T$ is the total number of noising steps, the sample has been approximately degraded into the standard unit Gaussian distribution. In summary, over a period of $T$ steps, a sample of data $\mathbf{x} \sim p_{data}\left(\mathbf{x}\right)$ is transformed into a sample from a unit Gaussian, $\mathbf{z}_T \sim \mathcal{N}\left(0, \mathbf{I}\right)$. Noised variables are denoted by $\mathbf{z}$.

Forward process transition kernels take the form:
$$q(\mathbf{z}_t | \mathbf{z}_{t-1}) = \mathcal{N}(\mathbf{z}_t; \sqrt{\alpha_t} \mathbf{z}_{t-1}, (1 - \alpha_t) \mathbf{I}),$$

With a cumulative forward noising to an arbitrary $t$ accomplished via:

$$q(\mathbf{z}_t | \mathbf{x}) = \mathcal{N}(\mathbf{z}_t; \sqrt{\bar{\alpha}_t} \mathbf{x}, (1 - \bar{\alpha}_t) \mathbf{I}),$$

exploiting the property that the product of two Gaussian distributions is a Gaussian distribution. During training, $t$ is sampled uniformly, and over training iterations, the noise-predicting neural network learns to predict noise at every level of noising:

$$t \sim U(0, T), \qquad \mathbf{z}_t = \sqrt{\bar{\alpha_t}} \mathbf{z}_{t-1} + \sqrt{1 - \bar{\alpha_t}} \epsilon$$

> Typically, the forward process can have anywhere between 1000 and 2000 steps. 
---

#### **Reverse process**
Bayes' theorem is used to formulate the conditional distribution for a step backwards in the process:
$$q\left(\mathbf{z}_{t-1} | \mathbf{z}_t\right) = \frac{q\left(\mathbf{z}_{t-1} | \mathbf{z}_t\right)q\left(\mathbf{z}_{t-1}\right)}{q\left(\mathbf{z}_t\right)},$$

which simplifies to the following form:

$$q(\mathbf{z}_{t-1} | \mathbf{z}_t, \mathbf{x}) = \mathcal{N}\left(\mathbf{z}_{t-1}; \underbrace{\left(\mathbf{z}_t\frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{(1 - \bar{\alpha}_t)} + \mathbf{x}\frac{\sqrt{\bar{\alpha}_{t-1}}(1 - \alpha_t)}{(1 - \bar{\alpha}_t)}\right)}_{\textcolor{red}{\mu}}, \underbrace{\frac{(1 - \alpha_t)(1 - \bar{\alpha}_{t-1}}{(1 - \bar{\alpha}_t)}\mathbf{I}}_{\textcolor{red}{\sigma^2}} \right)$$

where we arrive at a problem. The $\mu$ term contains $\mathbf{x}$, a sample from the original dataset. In order to generate a new sample from noise, it is clear that a dependency on a sample from the data is not desirable. DDPMs solve this issue by approximating $\mu$ via the noise-predicting neural network. 

> Key point: the forward process is **prescribed**, and the reverse process is **learnt** via parameterisation -- covered below. 

---

#### **Training objective**

A full derivation of the training objective is available in the cited work, however there are **three** candidate training objectives:

1. $\textcolor{red}{\mu}$ -- the mean of the reverse process.
2. $\textcolor{red}{\mathbf{x}}$ -- the original, noise-free sample from the original data distribution.
3. $\textcolor{red}{\epsilon}$ -- the unscaled unit Gaussian noise added to the noisy data sample.

In the previous segments, the neural network which forms the backbone of a diffusion model has been described as a noise-predicting neural network. The reason for this is that typically, the third objective is used, from empirical results showing that sample quality of the trained DDPM is superior when trained with this objective. 

---

### 1.2 Continuous-time formulation

In this section, the continuous time formulation (<cite>[Kingma et al., 2023][3]</cite>) is provided. This formulation provides faster training convergence, and robustness to changes in noising schedule. 

Why continuous-time? In the original DDPM formulation, the noising 'timesteps' are defined discretely. In the continuous formulation, rather than prescribing fixed discrete values for additive noise, the problem is **(1)**: reformulated to be in terms of $\lambda = log(\textrm{Signal/noise ratio}),$ -- which is continuous -- and **(2)**: defined as random samples within fixed upper and lower bounds of $\lambda$. This allows for a richer representation of noising states, and decouples the noising process from fixed steps of additive noise, meaning that a smaller number of steps in the reverse process can yield a high quality generated sample. 

> This formulation reduces the number of iterations required to generate a sample from DDPM by an order of magnitude. 

Recent work has reduced the iteration cost of DDPMs further. For further work in this area, the reader is referred to the work of <cite>[Song et al., 2022][4]</cite>. 

### 1.3 Pros and Cons

The general pros and cons of different generative machine learning methods are well laid out in <cite>[Zhao et al., 2022][5]</cite>, where the generative method problem is laid out as a `trilemma'. Here we list a few general pros and cons of DDPMs.

#### 1.3.1 Pros

- Effectively infinite sampling possibilities in-distribution
- No mode-collapse: the lack of adversarial training removes the likelihood of exploitative generative samples dominating the output space.
- Training convergence is smooth.

#### 1.3.2 Cons

- Generative procedure requires iteration -- although recent work has made significant strides in reducing this (see <cite>[Song et al., 2022][6]</cite> for further details)
- Difficulties in adherence to admissible space
- Poor out-of-distribution performance. 


[1]: https://arxiv.org/pdf/2006.11239
[2]: https://arxiv.org/pdf/1907.05600
[3]: https://arxiv.org/pdf/2107.00630
[4]: https://arxiv.org/pdf/2010.02502
[5]: https://arxiv.org/pdf/2112.07804
[6]: https://arxiv.org/pdf/2010.02502

## 2. Case setup

This tutorial will focus on the 2D forced turbulence problem, 

$$ \frac{\partial u_i}{\partial t} + u_j \dfrac{\partial u_i}{\partial x_j} = - \dfrac{\partial p}{\partial x_i} + \frac{1}{\mathrm{Re}} \dfrac{\partial^2 u_i}{\partial x_j \partial x_j} + f_i,$$

$$\dfrac{\partial u_i}{\partial x_i}  = 0,$$

where $u_i$ is the dimensionless velocity field, $x_i$ is the spatial coordinate, $p$ is the dimensionless pressure, $\mathrm{Re}$ is the Reynolds number, $f_i = \sin{\left(10 \delta_{i2} x_1\right)}$ is a steady sinusoidal forcing term, and $\delta$ is the Kronecker delta. The domain is taken as a square of length $2 \pi$ in physical space, which for $128\times128$ grid points leads to a square of $64\times64$ in wavenumber space. The boundaries are fully periodic. Simulations are carried out for $\textrm{Re=222}$. Data is downsampled from $1024\times1024$, which is the original DNS resolution. 

### 2.1 Imports and Data loading

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from typing import Callable
import einops
import matplotlib.pyplot as plt
import requests
from pathlib import Path
import zipfile

In [3]:
# Zenodo record
record_id = 22031070
api = f"https://zenodo.org/api/records/{record_id}"

# Files to download
files_to_download = [
    "nu_0p0045_256_5k_8x_normalised.pt",
    "nu_0p0045_256_8x_val_normalised.pt",
]

# Create data directory
data_dir = Path()
data_dir.mkdir(exist_ok=True)

# Get Zenodo metadata
metadata = requests.get(api)
metadata.raise_for_status()
metadata = metadata.json()

# Download each file
for filename in files_to_download:

    # Find the file in the Zenodo record
    for f in metadata["files"]:
        if f["key"] == filename:
            download_url = f["links"]["self"]
            break
    else:
        raise RuntimeError(
            f"{filename} not found in Zenodo record {record_id}."
        )

    # Local file path
    file_path = data_dir / filename

    # Download if it doesn't already exist
    if not file_path.exists():
        print(f"Downloading {filename} ...")

        with requests.get(download_url, stream=True) as r:
            r.raise_for_status()

            with open(file_path, "wb") as fp:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        fp.write(chunk)

        print(f"Download completed: {file_path}")

    else:
        print(f"File already exists: {file_path}")

print("\nAll files are ready:")
for filename in files_to_download:
    print(data_dir / filename)

KeyboardInterrupt: 

In [ ]:
train_dir = 'nu_0p0045_256_5k_8x_normalised.pt'
val_dir = 'nu_0p0045_256_8x_val_normalised.pt'

In [ ]:
train_vel = torch.load(train_dir)[1]
val_vel = torch.load(val_dir)[1]

In [ ]:
train_vel.shape, val_vel.shape

> train and val are 4D tensors containing high-resolution snapshots of temporally decorrelated flow states. Their shape is [batch, channel, height, width] -- batch is the number of smaples (snapshots), channel is velocity components (u, v), height and width are the grid resolution. Below is a short visualisation of five random samples from the training set, shown for u. 

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(15, 3))
for ax in axs:  
    i = torch.randint(0, train_vel.shape[0], (1,))
    im = ax.imshow(train_vel[i, 0].squeeze(), cmap='RdBu_r')
    ax.axis('off')
cbar = fig.colorbar(im, ax=axs, fraction=0.02, pad=0.04)
cbar.set_label("$u$")
plt.show()

#### Data transforms

For tractability, the DNS data will be:

1. Converted from velocity fields, to vorticity fields.
2. Downsampled by a factor of four, to yield $64\times64$ snapshots of u, v. This is done directly via torch. 

In [ ]:
import numpy as np
import findiff

def compute_vorticity_batch(data, n_cells=256, acc=8):
    """
    Compute vorticity for batched velocity fields with periodic boundary conditions.
    
    Parameters:
    -----------
    data : np.ndarray
        Shape [N, 2, H, W] where N is number of snapshots,
        dimension 1 contains [u, v] velocity components
    n_cells : int
        Number of grid cells (assumes square domain)
    acc : int
        Accuracy order for finite differences
    
    Returns:
    --------
    omega : np.ndarray
        Vorticity field of shape [N, H, W]
    """
    # Grid spacing (matches original)
    dx = 2 * np.pi / n_cells * 4
    
    # Padding for stencil
    pad = acc // 2 + 2
    
    # Extract and pad with periodic BC
    u = np.pad(data[:, 0], ((0, 0), (pad, pad), (pad, pad)), mode='wrap')
    v = np.pad(data[:, 1], ((0, 0), (pad, pad), (pad, pad)), mode='wrap')
    
    # Finite differences
    d_dx = findiff.Diff(1, dx, acc=acc)
    d_dy = findiff.Diff(2, dx, acc=acc)
    
    # vorticity = dv/dx - du/dy, extract valid region
    return d_dx(v)[:, pad:-pad, pad:-pad] - d_dy(u)[:, pad:-pad, pad:-pad]

In [ ]:
# Calculating vorticity
train_omega = torch.tensor(compute_vorticity_batch(train_vel)).unsqueeze(1)
val_omega = torch.tensor(compute_vorticity_batch(val_vel)).unsqueeze(1)
print(train_omega.shape)
print(val_omega.shape)

In [ ]:
# Downsampling by a factor of 4
train = torch.nn.functional.interpolate(train_omega, scale_factor=0.25)
val = torch.nn.functional.interpolate(val_omega, scale_factor=0.25)
print(train.shape)
print(val.shape)

In [ ]:
# vorticity contours of five random snapshots(smaples) from the training set
fig, axs = plt.subplots(1, 5, figsize=(15, 3))
for ax in axs:
    i = torch.randint(low=0, high=train.shape[0], size=[1])
    im = ax.imshow(train[i, 0].squeeze(), cmap='twilight')
    ax.axis('off')
cbar = fig.colorbar(im, ax=axs, fraction=0.02, pad=0.04)
cbar.set_label("$\omega$")
plt.show()

## 3. ML Model Components

The actual DDPM model is comprised of three parts:

1. The noise-predicting neural network.
2. The forward process of additive noise -- used during training.
3. The reverse process of noise removal -- used during inference, i.e. generation of new samples.

Each of these components is presented separately in the following subsections. 

### 3.1 Noise-predicting neural network

As has been described in previous sections, there are several training objectives which can be used to parameterise the mean of the reverse process. The entire goal is to have a fully parameterised reverse process in order to be able to tractably compute the prior on Gaussian noise, and generate new samples. In practice, the noise-predicting neural network is the preferred choice as empirical results have shown that generated samples are of a superior quality from models trained with this objective.

In [ ]:
"""
Noise-Predicting U-Net for Continuous-Time Diffusion Models
============================================================

This module implements a U-Net architecture designed to predict the noise component
in continuous-time Denoising Diffusion Probabilistic Models (DDPMs). The network
takes a noisy input and a noise level (continuous time parameter) and predicts
the noise that was added to the clean data.

Background: Continuous-Time DDPMs
---------------------------------
In continuous-time diffusion models, the forward process is defined by an SDE:

    dx = f(x, t)dt + g(t)dW
where f is the drift, g is the diffusion coefficient, and W is a Wiener process.
The reverse process requires learning the score function ∇_x log p_t(x), which
is equivalent to learning to predict the noise ε that was added at each timestep.
This U-Net serves as the noise predictor ε_θ(x_t, t).

Architecture Overview
---------------------
The U-Net follows the standard encoder-decoder structure with skip connections:

1. **Encoder (Downsampling path)**: Progressively reduces spatial resolution
   while increasing channel depth, capturing multi-scale features.

2. **Bottleneck**: Processes the most compressed representation with
   self-attention for global context.

3. **Decoder (Upsampling path)**: Reconstructs spatial resolution using
   skip connections from the encoder to preserve fine-grained details.

Key Features:
- Sinusoidal positional encoding for continuous noise levels
- Feature-wise affine transformations for noise conditioning
- Self-attention at specified resolutions for global context
- Periodic padding support for turbulence/flow applications
- GroupNorm for stable training with small batch sizes
"""
from __future__ import annotations
import math
import functools as ft
from inspect import isfunction
from typing import Callable, Optional, Sequence
import torch
from torch import nn, Tensor
import torch.nn.functional as F

# Utility Functions
def exists(x: Optional[object]) -> bool:
    """Check if a value is not None.
    
    Parameters
    ----------
    x : Optional[object]
        Value to check.
        
    Returns
    -------
    bool
        True if x is not None, False otherwise.
    """
    return x is not None


def default(val: Optional[object], d: object | Callable[[], object]) -> object:
    """Return val if it exists, otherwise return default value d.
    
    If d is a callable (function), it will be called to produce the default.
    This lazy evaluation is useful for expensive default computations.
    
    Parameters
    ----------
    val : Optional[object]
        Primary value to return if not None.
    d : object | Callable[[], object]
        Default value or callable that produces default value.
        
    Returns
    -------
    object
        val if not None, otherwise d() if d is callable, else d.
        
    Examples
    --------
    >>> default(5, 10)
    5
    >>> default(None, 10)
    10
    >>> default(None, lambda: expensive_computation())  # Only called if needed
    """
    if exists(val):
        return val
    return d() if isfunction(d) else d


# Noise Level Embedding
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for continuous noise levels.
    
    This module encodes a scalar noise level (or continuous timestep) into a
    high-dimensional vector using sinusoidal functions at different frequencies.
    This is analogous to transformer positional encodings but applied to the
    diffusion timestep.
    
    The encoding uses the formula:
        PE(t, 2i)   = sin(t * exp(-log(10000) * i/d))
        PE(t, 2i+1) = cos(t * exp(-log(10000) * i/d))
    
    where t is the noise level, i is the dimension index, and d is half the
    embedding dimension. This creates a unique representation for each noise
    level that the network can learn to condition on.
    
    Parameters
    ----------
    dim : int
        Output embedding dimension. Must be even (split between sin and cos).
        
    Notes
    -----
    - Higher frequencies (small i) capture fine-grained noise level differences
    - Lower frequencies (large i) capture coarse noise level information
    - The exponential spacing ensures good coverage across noise levels
    
    References
    ----------
    Adapted from WaveGrad: https://github.com/lmnt-com/wavegrad
    """
    
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.dim = dim

    def forward(self, noise_level: Tensor) -> Tensor:
        """Encode noise levels into sinusoidal embeddings.
        
        Parameters
        ----------
        noise_level : Tensor
            Noise levels to encode, shape (batch_size,).
            Values typically in [0, 1] for continuous-time diffusion.
            
        Returns
        -------
        Tensor
            Positional encodings, shape (batch_size, dim).
        """
        # Number of frequency components (half for sin, half for cos)
        count = self.dim // 2
        
        # Create frequency steps: exp(-log(10000) * i/count) for i in [0, count)
        # This gives frequencies from 1 to 1/10000
        step = torch.arange(count, dtype=noise_level.dtype,
                            device=noise_level.device) / count
        
        # Compute arguments for sin/cos: noise_level * frequency
        # Shape: (batch_size, 1) * (1, count) -> (batch_size, count)
        encoding = noise_level.unsqueeze(1) * torch.exp(-math.log(1e4) * step.unsqueeze(0))
        
        # Concatenate sin and cos components
        encoding = torch.cat([torch.sin(encoding), torch.cos(encoding)], dim=-1)
        
        return encoding


# Noise Conditioning Layers

class FeatureWiseAffine(nn.Module):
    """Feature-wise affine transformation conditioned on noise level.
    
    This module injects noise level information into feature maps using either:
    1. Additive conditioning: x' = x + MLP(noise_embed)
    2. Affine conditioning: x' = (1 + γ) * x + β, where γ, β = MLP(noise_embed)
    
    The affine version (FiLM - Feature-wise Linear Modulation) provides more
    expressive conditioning by allowing the network to scale features based
    on the noise level, which is particularly useful for diffusion models
    where the signal-to-noise ratio varies significantly across timesteps.
    
    Parameters
    ----------
    in_channels : int
        Dimension of the noise embedding input.
    out_channels : int
        Number of feature map channels to modulate.
    use_affine_level : bool, default=False
        If True, use full affine (scale + shift). If False, use additive only.
        
    References
    ----------
    Perez et al. (2018): "FiLM: Visual Reasoning with a General Conditioning Layer"
    """
    
    def __init__(
        self, 
        in_channels: int, 
        out_channels: int, 
        use_affine_level: bool = False
    ) -> None:
        super().__init__()
        self.use_affine_level = use_affine_level
        
        # MLP that maps noise embedding to modulation parameters
        # Output size is out_channels for additive, 2*out_channels for affine
        self.noise_func = nn.Sequential(
            nn.Linear(in_channels, out_channels * (1 + self.use_affine_level))
        )

    def forward(self, x: Tensor, noise_embed: Tensor) -> Tensor:
        """Apply noise-conditioned modulation to feature maps.
        
        Parameters
        ----------
        x : Tensor
            Input feature maps, shape (batch, channels, height, width).
        noise_embed : Tensor
            Noise level embedding, shape (batch, embed_dim).
            
        Returns
        -------
        Tensor
            Modulated feature maps, same shape as input.
        """
        batch = x.shape[0]
        
        if self.use_affine_level:
            # Affine modulation: (1 + γ) * x + β
            # Split MLP output into scale (gamma) and shift (beta)
            gamma, beta = self.noise_func(noise_embed).view(
                batch, -1, 1, 1
            ).chunk(2, dim=1)
            x = (1 + gamma) * x + beta
        else:
            # Simple additive conditioning
            x = x + self.noise_func(noise_embed).view(batch, -1, 1, 1)
            
        return x


# Activation Functions

class Swish(nn.Module):
    """Swish activation function: f(x) = x * sigmoid(x).
    
    Also known as SiLU (Sigmoid Linear Unit). Swish is a smooth, non-monotonic
    activation that has been shown to outperform ReLU in many deep networks,
    particularly in diffusion models.
    
    Properties:
    - Smooth and differentiable everywhere
    - Non-monotonic: has a small negative region
    - Self-gated: output depends on both input magnitude and sign
    - Approaches identity for large positive inputs
    - Approaches zero for large negative inputs
    
    References
    ----------
    Ramachandran et al. (2017): "Searching for Activation Functions"
    """
    
    def forward(self, x: Tensor) -> Tensor:
        """Apply Swish activation.
        
        Parameters
        ----------
        x : Tensor
            Input tensor of any shape.
            
        Returns
        -------
        Tensor
            Activated tensor, same shape as input.
        """
        return x * torch.sigmoid(x)


# Resampling Layers (for Periodic Domains)

class PeriodicUpsampler(nn.Module):
    """Upsampler with periodic boundary handling for turbulence applications.
    
    Standard upsampling can introduce boundary artifacts in periodic domains
    (e.g., turbulent flow fields on a torus). This module handles periodicity
    by padding the input with wrapped values before upsampling, then cropping
    to the target size.
    
    Process:
    1. Pad input with circular (periodic) boundary conditions
    2. Apply standard upsampling (nearest neighbour or bilinear)
    3. Crop to remove padding artifacts at boundaries
    
    This ensures smooth transitions across periodic boundaries in the
    upsampled output, which is critical for physical consistency in
    flow field generation.
    
    Parameters
    ----------
    scale_factor : int, default=2
        Upsampling factor (2 means double the resolution).
    mode : str, default='nearest'
        Upsampling interpolation mode ('nearest' or 'bilinear').
    npad : int, default=2
        Number of cells to pad before upsampling. Larger values provide
        smoother periodic transitions but increase computation.
    """
    
    def __init__(
        self, 
        scale_factor: int = 2, 
        mode: str = 'nearest', 
        npad: int = 2
    ) -> None:
        super().__init__()
        
        self.mode = mode
        self.scale_factor = scale_factor
        self.npad = npad
        
        # Circular padding function (wraps values from opposite edge)
        self.fn_pad = ft.partial(
            F.pad, 
            mode='circular', 
            pad=tuple(self.npad for _ in range(4))  # (left, right, top, bottom)
        )
        
        # Standard upsampler
        self.upsampler = nn.Upsample(scale_factor=self.scale_factor, mode=self.mode)
        
        # Compute slice to crop upsampled padding
        upsampled_pad = self.scale_factor * self.npad
        self.slice = slice(upsampled_pad, -upsampled_pad)

    def forward(self, x: Tensor) -> Tensor:
        """Upsample with periodic boundary handling.
        
        Parameters
        ----------
        x : Tensor
            Low resolution tensor, shape (batch, channels, H, W).
            
        Returns
        -------
        Tensor
            Upsampled tensor, shape (batch, channels, H*scale, W*scale).
        """
        # Step 1: Circular padding
        x = self.fn_pad(x)
        
        # Step 2: Standard upsampling
        x = self.upsampler(x)
        
        # Step 3: Crop to original domain (scaled)
        x = x[..., self.slice, self.slice]
        
        return x


class Upsample(nn.Module):
    """Upsampling block with periodic handling and learned convolution.
    
    Combines periodic-aware upsampling with a convolutional layer to allow
    the network to learn how to best interpolate features at higher resolution.
    
    Parameters
    ----------
    dim : int
        Number of input/output channels.
    """
    
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.up = PeriodicUpsampler()
        self.conv = nn.Conv2d(dim, dim, kernel_size=3, padding=1)

    def forward(self, x: Tensor) -> Tensor:
        """Upsample and refine with convolution.
        
        Parameters
        ----------
        x : Tensor
            Input tensor, shape (batch, dim, H, W).
            
        Returns
        -------
        Tensor
            Upsampled tensor, shape (batch, dim, 2H, 2W).
        """
        return self.conv(self.up(x))


class Downsample(nn.Module):
    """Downsampling block using strided convolution.
    
    Reduces spatial resolution by factor of 2 using a learned strided
    convolution rather than pooling. This allows the network to learn
    which information to preserve during downsampling.
    
    Parameters
    ----------
    dim : int
        Number of input/output channels.
    conv_kwargs : dict
        Additional convolution arguments (e.g., padding_mode for periodicity).
    """
    
    def __init__(self, dim: int, conv_kwargs: dict) -> None:
        super().__init__()
        # stride=2 for 2x downsampling, padding=1 to maintain correct output size
        self.conv = nn.Conv2d(dim, dim, kernel_size=3, stride=2, padding=1)

    def forward(self, x: Tensor) -> Tensor:
        """Downsample by factor of 2.
        
        Parameters
        ----------
        x : Tensor
            Input tensor, shape (batch, dim, H, W).
            
        Returns
        -------
        Tensor
            Downsampled tensor, shape (batch, dim, H//2, W//2).
        """
        return self.conv(x)


# Core Building Blocks

class Block(nn.Module):
    """Basic convolutional block with normalisation and activation.
    
    Applies the sequence: GroupNorm -> Swish -> Dropout -> Conv2d
    
    GroupNorm is preferred over BatchNorm in diffusion models because:
    1. Works well with small batch sizes (common in high-res generation)
    2. More stable training dynamics
    3. No dependence on batch statistics during inference
    
    Parameters
    ----------
    dim : int
        Number of input channels.
    dim_out : int
        Number of output channels.
    conv_kwargs : dict
        Convolution arguments (e.g., padding, padding_mode).
    groups : int, default=32
        Number of groups for GroupNorm.
    dropout : float, default=0
        Dropout probability. Set to 0 to disable.
    """
    
    def __init__(
        self, 
        dim: int, 
        dim_out: int, 
        conv_kwargs: dict, 
        groups: int = 32, 
        dropout: float = 0
    ) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(groups, dim),
            Swish(),
            nn.Dropout(dropout) if dropout != 0 else nn.Identity(),
            nn.Conv2d(dim, dim_out, kernel_size=3, **conv_kwargs)
        )

    def forward(self, x: Tensor) -> Tensor:
        """Apply normalisation, activation, and convolution.
        
        Parameters
        ----------
        x : Tensor
            Input tensor, shape (batch, dim, H, W).
            
        Returns
        -------
        Tensor
            Output tensor, shape (batch, dim_out, H, W).
        """
        return self.block(x)


class ResnetBlock(nn.Module):
    """Residual block with noise level conditioning.
    
    Implements a pre-activation residual block with noise conditioning injected
    between the two convolutional layers. The residual connection allows
    gradients to flow directly through the network, improving training of
    deep architectures.
    
    Structure::
    
        x -> [Block1] -> [NoiseCondition] -> [Block2] -> (+) -> out
        |__________________________________________________|
                            [ResConv]
    
    Parameters
    ----------
    dim : int
        Number of input channels.
    dim_out : int
        Number of output channels.
    conv_kwargs : dict
        Convolution arguments for periodic padding etc.
    noise_level_emb_dim : Optional[int]
        Dimension of noise level embedding. None disables conditioning.
    dropout : float, default=0
        Dropout probability.
    use_affine_level : bool, default=False
        If True, use affine (scale+shift) noise conditioning.
    norm_groups : int, default=32
        Number of groups for GroupNorm.
    """
    
    def __init__(
        self, 
        dim: int, 
        dim_out: int, 
        conv_kwargs: dict,
        noise_level_emb_dim: Optional[int] = None, 
        dropout: float = 0, 
        use_affine_level: bool = False, 
        norm_groups: int = 32
    ) -> None:
        super().__init__()
        
        # Noise conditioning module
        self.noise_func = FeatureWiseAffine(
            noise_level_emb_dim, dim_out, use_affine_level
        )
        
        # Two convolutional blocks
        self.block1 = Block(dim, dim_out, conv_kwargs, groups=norm_groups)
        self.block2 = Block(dim_out, dim_out, conv_kwargs, groups=norm_groups, dropout=dropout)
        
        # Residual projection (1x1 conv if dimensions change, identity otherwise)
        self.res_conv = (
            nn.Conv2d(dim, dim_out, kernel_size=1, **conv_kwargs) 
            if dim != dim_out 
            else nn.Identity()
        )

    def forward(self, x: Tensor, time_emb: Tensor) -> Tensor:
        """Apply residual block with noise conditioning.
        
        Parameters
        ----------
        x : Tensor
            Input tensor, shape (batch, dim, H, W).
        time_emb : Tensor
            Noise level embedding, shape (batch, noise_level_emb_dim).
            
        Returns
        -------
        Tensor
            Output tensor, shape (batch, dim_out, H, W).
        """
        # First convolution block
        h = self.block1(x)
        
        # Inject noise level information
        h = self.noise_func(h, time_emb)
        
        # Second convolution block
        h = self.block2(h)
        
        # Add residual connection
        return h + self.res_conv(x)


# Self-Attention

class SelfAttention(nn.Module):
    """Multi-head self-attention for capturing global dependencies.
    
    Attention allows the network to model long-range spatial relationships,
    which is particularly important for generating coherent large-scale
    structures in images or turbulent flow fields.
    
    In diffusion models, attention is typically applied only at lower
    resolutions (e.g., 8x8, 16x16) to balance expressiveness with
    computational cost.
    
    Parameters
    ----------
    in_channel : int
        Number of input/output channels.
    n_head : int, default=1
        Number of attention heads. Multi-head attention allows the model
        to attend to different aspects of the input simultaneously.
    norm_groups : int, default=32
        Number of groups for GroupNorm applied before attention.
        
    Notes
    -----
    Uses einsum notation for clarity in tensor operations:
    - 'bnchw, bncyx -> bnhwyx': Query-Key dot product
    - 'bnhwyx, bncyx -> bnchw': Attention-Value multiplication
    """
    
    def __init__(
        self, 
        in_channel: int, 
        n_head: int = 1, 
        norm_groups: int = 32
    ) -> None:
        super().__init__()
        
        self.n_head = n_head
        
        # Pre-attention normalisation
        self.norm = nn.GroupNorm(norm_groups, in_channel)
        
        # Project to queries, keys, and values simultaneously
        self.qkv = nn.Conv2d(in_channel, in_channel * 3, kernel_size=1, bias=False)
        
        # Output projection
        self.out = nn.Conv2d(in_channel, in_channel, kernel_size=1)

    def forward(self, input: Tensor) -> Tensor:
        """Apply self-attention with residual connection.
        
        Parameters
        ----------
        input : Tensor
            Input tensor, shape (batch, channel, height, width).
            
        Returns
        -------
        Tensor
            Output tensor with attention applied, same shape as input.
        """
        batch, channel, height, width = input.shape
        n_head = self.n_head
        head_dim = channel // n_head
        
        # Normalise input
        norm = self.norm(input)
        
        # Compute Q, K, V projections
        # Shape: (batch, 3*channel, H, W) -> (batch, n_head, 3*head_dim, H, W)
        qkv = self.qkv(norm).view(batch, n_head, head_dim * 3, height, width)
        query, key, value = qkv.chunk(3, dim=2)  # Each: (batch, n_head, head_dim, H, W)
        
        # Compute attention scores: Q @ K^T / sqrt(d)
        # For each spatial position (h,w), compute attention to all positions (y,x)
        attn = torch.einsum(
            "bnchw, bncyx -> bnhwyx", query, key
        ).contiguous() / math.sqrt(channel)
        
        # Softmax over spatial positions
        attn = attn.view(batch, n_head, height, width, -1)
        attn = torch.softmax(attn, dim=-1)
        attn = attn.view(batch, n_head, height, width, height, width)
        
        # Apply attention to values
        out = torch.einsum("bnhwyx, bncyx -> bnchw", attn, value).contiguous()
        out = self.out(out.view(batch, channel, height, width))
        
        # Residual connection
        return out + input


class ResnetBlocWithAttn(nn.Module):
    """Residual block optionally followed by self-attention.
    
    Combines a ResNet block with optional self-attention. In the U-Net,
    attention is typically enabled only at specific resolutions (usually
    the lowest) to capture global structure without excessive computation.
    
    Parameters
    ----------
    dim : int
        Number of input channels.
    dim_out : int
        Number of output channels.
    conv_kwargs : dict
        Convolution arguments for periodic padding.
    noise_level_emb_dim : Optional[int]
        Noise embedding dimension for conditioning.
    norm_groups : int, default=32
        GroupNorm groups.
    dropout : float, default=0
        Dropout probability.
    with_attn : bool, default=False
        Whether to apply self-attention after the residual block.
    """
    
    def __init__(
        self, 
        dim: int, 
        dim_out: int, 
        conv_kwargs: dict,
        *, 
        noise_level_emb_dim: Optional[int] = None, 
        norm_groups: int = 32, 
        dropout: float = 0, 
        with_attn: bool = False
    ) -> None:
        super().__init__()
        self.with_attn = with_attn
        
        self.res_block = ResnetBlock(
            dim, dim_out, conv_kwargs, noise_level_emb_dim, 
            norm_groups=norm_groups, dropout=dropout
        )
        
        if with_attn:
            self.attn = SelfAttention(dim_out, norm_groups=norm_groups)

    def forward(self, x: Tensor, time_emb: Tensor) -> Tensor:
        """Apply residual block and optional attention.
        
        Parameters
        ----------
        x : Tensor
            Input tensor, shape (batch, dim, H, W).
        time_emb : Tensor
            Noise level embedding.
            
        Returns
        -------
        Tensor
            Output tensor, shape (batch, dim_out, H, W).
        """
        x = self.res_block(x, time_emb)
        if self.with_attn:
            x = self.attn(x)
        return x


# Main U-Net Architecture

class UNet(nn.Module):
    """Noise-predicting U-Net for continuous-time diffusion models.
    
    This U-Net takes a noisy sample x_t and the noise level t, and predicts
    the noise ε that was added. The architecture uses:
    
    - **Encoder**: Progressive downsampling with increasing channels
    - **Bottleneck**: Low-resolution processing with self-attention
    - **Decoder**: Progressive upsampling with skip connections from encoder
    - **Noise conditioning**: Injected at every ResNet block
    
    For turbulence applications, periodic padding is used throughout to
    respect the periodic boundary conditions of the physical domain.
    
    Architecture Diagram::
    
        Input (x_t, t)
            |
            v
        +------------------+
        |  Noise Level     |------------------------------+
        |  Embedding       |                              |
        +------------------+                              |
            |                                             |
            v                                             v
        +------------------+                     (conditioned at each
        |  Initial Conv    |                      ResNet block)
        +------------------+
            |
            v
        +------------------+
        |  Encoder         |------ Skip connections ------+
        |  (Downsampling)  |                              |
        +------------------+                              |
            |                                             |
            v                                             |
        +------------------+                              |
        |  Bottleneck      |                              |
        |  (with Attn)     |                              |
        +------------------+                              |
            |                                             |
            v                                             |
        +------------------+                              |
        |  Decoder         |<-----------------------------+
        |  (Upsampling)    |
        +------------------+
            |
            v
        +------------------+
        |  Final Conv      |
        +------------------+
            |
            v
        Output (predicted noise)
    
    Parameters
    ----------
    in_channel : int, default=6
        Number of input channels. For super-resolution, this might be
        low-res input + noisy high-res concatenated.
    out_channel : int, default=3
        Number of output channels (predicted noise dimensions).
    inner_channel : int, default=32
        Base channel count. Actual channels at each level are
        inner_channel * channel_mults[level].
    norm_groups : int, default=32
        Number of groups for GroupNorm throughout the network.
    channel_mults : tuple[int, ...], default=(1, 2, 4, 8, 8)
        Channel multipliers for each resolution level. Defines both
        the number of levels and the channel progression.
    attn_res : Sequence[int], default=[8]
        Resolutions at which to apply self-attention. E.g., [8] applies
        attention when the spatial size is 8x8.
    res_blocks : int, default=3
        Number of residual blocks per resolution level.
    dropout : float, default=0.2
        Dropout probability for regularisation.
    with_noise_level_emb : bool, default=True
        Whether to use noise level embeddings. Should be True for
        standard diffusion training.
    image_size : tuple[int, int], default=(128, 128)
        Input spatial dimensions. Used to determine when to apply attention.
    periodic : bool, default=True
        Whether to use periodic (circular) padding. Essential for
        turbulence applications.
        
    Examples
    --------
    >>> # Create U-Net for 128x128 turbulence super-resolution
    >>> model = UNet(
    ...     in_channel=6,      # 3 low-res + 3 noisy high-res
    ...     out_channel=3,     # 3 velocity components
    ...     inner_channel=64,
    ...     channel_mults=(1, 2, 4, 8),
    ...     image_size=(128, 128),
    ...     periodic=True
    ... )
    >>> 
    >>> # Forward pass
    >>> x_t = torch.randn(4, 6, 128, 128)  # Batch of 4
    >>> t = torch.rand(4)                   # Noise levels in [0, 1]
    >>> noise_pred = model(x_t, t)          # Shape: (4, 3, 128, 128)
    """
    
    def __init__(
        self,
        in_channel: int = 1,
        out_channel: int = 1,
        inner_channel: int = 8,
        norm_groups: int = 8,
        channel_mults: tuple[int, ...] = (1, 2, 4, 8, 8),
        attn_res: Sequence[int] = [8],
        res_blocks: int = 2,
        dropout: float = 0.2,
        with_noise_level_emb: bool = True,
        image_size: tuple[int, int] = (64, 64),
        periodic: bool = True
    ) -> None:
        super().__init__()
        
        # Store configuration
        self.image_size = image_size
        self.in_channel = in_channel
        self.out_channel = out_channel
        self.shape = (1, out_channel, *image_size)
        
        # Configure convolution kwargs for periodic domains
        if periodic:
            _conv_kwargs = dict(padding='same', padding_mode='circular')
        else:
            _conv_kwargs = dict(padding=1)
        
        # Noise Level Embedding MLP
        # Maps scalar noise level to a rich embedding that conditions the network
        if with_noise_level_emb:
            noise_level_channel = inner_channel
            self.noise_level_mlp = nn.Sequential(
                PositionalEncoding(inner_channel),       # Sinusoidal encoding
                nn.Linear(inner_channel, inner_channel * 4),  # Expand
                Swish(),
                nn.Linear(inner_channel * 4, inner_channel)   # Project back
            )
        else:
            noise_level_channel = None
            self.noise_level_mlp = None
        
        # Encoder (Downsampling Path)
        num_mults = len(channel_mults)
        pre_channel = inner_channel
        feat_channels = [pre_channel]  # Track channels for skip connections
        now_res = image_size[0]        # Track current resolution for attention
        
        # Initial projection from input channels
        downs = [nn.Conv2d(in_channel, inner_channel, kernel_size=3, padding=1)]
        
        # Build encoder levels
        for ind in range(num_mults):
            is_last = (ind == num_mults - 1)
            use_attn = (now_res in attn_res)
            channel_mult = inner_channel * channel_mults[ind]
            
            # Residual blocks at this resolution
            for _ in range(res_blocks):
                downs.append(ResnetBlocWithAttn(
                    pre_channel, channel_mult,
                    conv_kwargs=_conv_kwargs,
                    noise_level_emb_dim=noise_level_channel,
                    norm_groups=norm_groups,
                    dropout=dropout,
                    with_attn=use_attn
                ))
                feat_channels.append(channel_mult)
                pre_channel = channel_mult
            
            # Downsample (except at last level)
            if not is_last:
                downs.append(Downsample(pre_channel, _conv_kwargs))
                feat_channels.append(pre_channel)
                now_res = now_res // 2
                
        self.downs = nn.ModuleList(downs)
        
        # Bottleneck
        # Process at lowest resolution with attention for global context
        self.mid = nn.ModuleList([
            ResnetBlocWithAttn(
                pre_channel, pre_channel, _conv_kwargs,
                noise_level_emb_dim=noise_level_channel,
                norm_groups=norm_groups,
                dropout=dropout,
                with_attn=True  # Attention at bottleneck
            ),
            ResnetBlocWithAttn(
                pre_channel, pre_channel, _conv_kwargs,
                noise_level_emb_dim=noise_level_channel,
                norm_groups=norm_groups,
                dropout=dropout,
                with_attn=False
            )
        ])
        
        # Decoder (Upsampling Path)
        ups = []
        
        # Build decoder levels (reverse order of encoder)
        for ind in reversed(range(num_mults)):
            is_last = (ind < 1)
            use_attn = (now_res in attn_res)
            channel_mult = inner_channel * channel_mults[ind]
            
            # Residual blocks at this resolution
            # Note: res_blocks+1 because we concatenate skip connection
            for _ in range(res_blocks + 1):
                # Input channels = current + skip connection
                ups.append(ResnetBlocWithAttn(
                    pre_channel + feat_channels.pop(), channel_mult,
                    _conv_kwargs,
                    noise_level_emb_dim=noise_level_channel,
                    norm_groups=norm_groups,
                    dropout=dropout,
                    with_attn=use_attn
                ))
                pre_channel = channel_mult
            
            # Upsample (except at last level)
            if not is_last:
                ups.append(Upsample(pre_channel))
                now_res = now_res * 2
                
        self.ups = nn.ModuleList(ups)
        
        # ---------------------------------------------------------------------
        # Final Output Projection
        # ---------------------------------------------------------------------
        self.final_conv = Block(
            pre_channel, 
            default(out_channel, in_channel),
            _conv_kwargs,
            groups=norm_groups
        )

    def forward(self, x: Tensor, time: Tensor) -> Tensor:
        """Predict noise given noisy input and noise level.
        
        Parameters
        ----------
        x : Tensor
            Noisy input tensor, shape (batch, in_channel, H, W).
            For conditional generation (e.g., super-resolution), this
            typically concatenates the conditioning input with the
            noisy sample.
        time : Tensor
            Noise levels (continuous timesteps), shape (batch,).
            Values typically in [0, 1] where 0 is clean and 1 is pure noise.
            
        Returns
        -------
        Tensor
            Predicted noise, shape (batch, out_channel, H, W).
            
        Notes
        -----
        The forward pass proceeds as:
        1. Encode noise level to embedding vector
        2. Pass through encoder, storing features for skip connections
        3. Process through bottleneck with attention
        4. Pass through decoder, concatenating skip features
        5. Final projection to output channels
        """
        # Compute noise level embedding
        t = self.noise_level_mlp(time) if exists(self.noise_level_mlp) else None
        
        # Encoder: collect features for skip connections
        feats = []
        for layer in self.downs:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(x, t)
            else:
                x = layer(x)
            feats.append(x)
        
        # Bottleneck
        for layer in self.mid:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(x, t)
            else:
                x = layer(x)
        
        # Decoder: use skip connections
        for layer in self.ups:
            if isinstance(layer, ResnetBlocWithAttn):
                # Concatenate skip connection and apply block
                x = layer(torch.cat((x, feats.pop()), dim=1), t)
            else:
                x = layer(x)
        
        # Final projection to output
        return self.final_conv(x)

### 3.2 Forward process


The following class contains all required functions to add noise to a sample, given the defined limits of log SNR. 

In [ ]:
"""
Continuous-Time Forward Diffusion Process

This module implements the forward (noising) process for continuous-time
diffusion models, parameterised by log signal-to-noise ratio (log-SNR or lambda).

Background: Log-SNR Parameterisation
------------------------------------
In continuous-time diffusion, the forward process adds noise according to:

    x_t = alpha(t) \dot x_0 + sigma(t) \dot epsilon,    where epsilon \sim N(0, I)

Rather than parameterising by time t in [0, 1], we use log-SNR:

    lambda(t) = log(alpha^2/sigma^2) = log(SNR)

This parameterisation has several advantages:
1. The noise schedule is defined implicitly via the SNR
2. Training is more stable across different noise levels
3. The loss weighting naturally emerges from the formulation

The relationship between alpha, sigma, and lambda is:
    alpha^2 = sigmoid(lambda) = 1 / (1 + exp(-lambda))
    sigma^2 = 1 - alpha^2 = sigmoid(-lambda)
"""

from __future__ import annotations

from functools import partial
from typing import Callable

import torch
import torch.nn as nn
from torch import Tensor


class ContinuousForwardDiffusion(nn.Module):
    """Forward diffusion process using log-SNR parameterisation.
    
    This class implements the forward (noising) process for training
    continuous-time diffusion models. It samples noise levels uniformly
    in log-SNR space and trains a denoising network to predict the
    added noise.
    
    The log-SNR, lambda, provides a natural parameterisation where:
    - Large lambda (e.g., +15): High SNR, nearly clean image
    - Small lambda (e.g., -10): Low SNR, nearly pure noise
    
    Parameters
    ----------
    denoise_model : Callable
        The denoising U-Net epsilon_theta(x_t, lambda) that predicts noise given
        a noisy sample and the log-SNR.
    lambda_min : float, default=-20
        Minimum log-SNR (maximum noise level). At lambda=-20, the signal
        is almost completely obscured by noise.
    lambda_max : float, default=20
        Maximum log-SNR (minimum noise level). At lambda=20, the image
        is nearly clean with minimal noise.
        
    Notes
    -----
    The default lambda range of [-20, 20] covers SNR from O(1e-7) to O(1e7),
    spanning from near-pure noise to near-clean images.
    
    Examples
    --------
    >>> model = UNet(in_channel=1, out_channel=1)
    >>> diffusion = ContinuousForwardDiffusion(model)
    >>> x = torch.randn(8, 1, 64, 64)  # Batch of clean images
    >>> x_noisy, eps_pred, eps_true = diffusion(x)
    >>> loss = F.mse_loss(eps_pred, eps_true)
    """
    
    def __init__(
        self, 
        denoise_model: Callable[[Tensor, Tensor], Tensor],
        lambda_min: float = -20,  
        lambda_max: float = 20
    ) -> None:
        super().__init__()
        self.lambda_min = lambda_min
        self.lambda_max = lambda_max
        self.denoise_model = denoise_model

    @staticmethod
    def _alpha(log_snr: Tensor) -> Tensor:
        """Compute signal coefficient alpha from log-SNR.
        
        From the definition lambda = log(alpha^2/sigma^2) and the constraint alpha^2 + sigma^2 = 1
        (variance-preserving diffusion), we derive:
        
            alpha^2 = sigmoid(lambda) = 1 / (1 + exp(-lambda))
            alpha = sqrt(sigmoid(lambda))
        
        Parameters
        ----------
        log_snr : Tensor
            Log signal-to-noise ratio lambda.
            
        Returns
        -------
        Tensor
            Signal coefficient alpha in (0, 1).
        """
        return torch.sqrt(1 / (1 + torch.exp(-log_snr)))

    @staticmethod
    def _sigma_2(alpha: Tensor) -> Tensor:
        """Compute noise variance sigma^2 from signal coefficient alpha.
        
        For variance-preserving diffusion: alpha^2 + sigma^2 = 1, hence sigma^2 = 1 - alpha^2.
        
        Parameters
        ----------
        alpha : Tensor
            Signal coefficient alpha.
            
        Returns
        -------
        Tensor
            Noise variance sigma^2.
        """
        return 1.0 - alpha ** 2

    def noising(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        """Apply forward diffusion to add noise at random log-SNR levels.
        
        Samples log-SNR values using stratified sampling across the batch
        for reduced variance, then applies the forward diffusion:
        
            x_noisy = alpha \dot x + sigma \dot epsilon,    where epsilon \sim N(0, I)
        
        The stratified sampling (from Variational Diffusion Models Appendix I.1) ensures that for
        a batch of size B, the sampled lambda values are spread evenly across
        [lambda_min, lambda_max] rather than clustering randomly.
        
        Parameters
        ----------
        x : Tensor
            Clean input tensor, shape (batch, channels, *spatial).
            
        Returns
        -------
        x_noisy : Tensor
            Noised tensor, same shape as input.
        log_snr_sample : Tensor
            Log-SNR values used for each sample, shape (batch,).
        noise : Tensor
            Gaussian noise that was added, same shape as input.
            
        Notes
        -----
        Stratified sampling: For batch size B, we sample u ~ U(0,1) once,
        then compute lambda_i from ((u + i/B) mod 1) for i = 1, ..., B.
        This ensures even coverage of the lambda range within each batch.
        """
        batch_size = x.shape[0]
        
        # Stratified sampling of log-SNR
        # Maps uniform samples to [lambda_min, lambda_max]
        scaling = partial(
            lambda u, l_max, l_min: u * (l_max - l_min) + l_min, 
            l_min=self.lambda_min, 
            l_max=self.lambda_max
        )
        
        # Sample single uniform, then create stratified samples across batch
        u = torch.rand((1,))
        stratified_u = torch.tensor(
            [((u + i / batch_size) % 1) for i in range(1, batch_size + 1)], 
            device=x.device
        )
        log_snr_sample = scaling(stratified_u)
        
        # Compute alpha and sigma, reshaping for broadcasting over spatial dims
        # Shape: (batch, 1, 1, ...) to broadcast with (batch, C, H, W, ...)
        n_spatial_dims = len(x.shape) - 1
        broadcast_shape = (-1,) + (1,) * n_spatial_dims
        
        alpha = self._alpha(log_snr_sample).view(*broadcast_shape)
        sigma = self._sigma_2(alpha).sqrt().view(*broadcast_shape)
        
        # Sample noise and apply forward diffusion
        noise = torch.randn_like(x, device=x.device)
        x_noisy = alpha * x + sigma * noise

        return x_noisy, log_snr_sample, noise

    def forward(self, x0: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        """Forward pass: noise the input and predict the noise.
        
        This is the main training step for diffusion models:
        1. Sample random noise levels (log-SNR)
        2. Add noise to clean images
        3. Predict the added noise with the denoising network
        
        The training loss is typically MSE between predicted and true noise:
            L = E[||epsilon_theta(x_t, lambda) - epsilon||^2]
        
        Parameters
        ----------
        x0 : Tensor
            Clean input images, shape (batch, channels, height, width).
            
        Returns
        -------
        x_noisy : Tensor
            Noised images at sampled noise levels.
        eps_hat : Tensor
            Predicted noise from the denoising network.
        eps : Tensor
            True noise that was added.
            
        Examples
        --------
        >>> diffusion = ContinuousForwardDiffusion(unet)
        >>> x_noisy, eps_pred, eps_true = diffusion(clean_images)
        >>> loss = F.mse_loss(eps_pred, eps_true)
        >>> loss.backward()
        """
        # Apply forward diffusion
        x_noisy, log_snr_sample, eps = self.noising(x0)

        # Predict noise (reshape log_snr for the network's expected input)
        eps_hat = self.denoise_model(x_noisy, log_snr_sample.view(-1, 1, 1, 1))

        return eps_hat, eps

### 3.3 Reverse process

This class contains all required components for unconditional generation. 

In [ ]:
"""
Continuous-Time Reverse Diffusion Process

This class implements the reverse (denoising/generation) process for
continuous-time diffusion models, using the log-SNR parameterisation.

Background: Reverse Process
---------------------------
The reverse process iteratively denoises a sample, starting from pure
Gaussian noise and progressively removing noise until a clean sample
is obtained. At each step, we:

1. Predict the noise in the current sample using the trained network
2. Estimate the clean image x0 from the noisy sample
3. Compute the posterior mean and variance for the previous timestep
4. Sample from this posterior to get the less-noisy sample

The discretisation uses a linear schedule in log-SNR space, stepping
from lambda_min (high noise) to lambda_max (low noise).
"""

from __future__ import annotations

from typing import Callable

import torch
import torch.nn as nn
from torch import Tensor
from torch.special import expm1
from tqdm import tqdm


class ContinuousReverseDiffusion(nn.Module):
    """Reverse diffusion process for sample generation.
    
    Implements the reverse (generative) process that transforms Gaussian
    noise into samples from the data distribution. The process is
    discretised into a fixed number of steps with a linear schedule
    in log-SNR space.
    
    The reverse process steps from lambda_min (nearly pure noise) to
    lambda_max (nearly clean), progressively denoising the sample at
    each step using the trained denoising network.
    
    Parameters
    ----------
    denoise_model : Callable[[Tensor, Tensor], Tensor]
        The trained denoising U-Net epsilon_theta(x_t, lambda) that predicts
        noise given a noisy sample and the log-SNR.
    num_steps : int
        Number of discretisation steps for the reverse process.
        Typical values: 64, 128, 256, 512. More steps generally yield
        higher quality but slower generation.
    lambda_min : float, default=-10
        Minimum log-SNR (starting point, high noise).
    lambda_max : float, default=15
        Maximum log-SNR (ending point, low noise).
        
    Attributes
    ----------
    lambdas : Tensor
        Linear schedule of log-SNR values, shape (num_steps,).
    alphas : Tensor
        Signal coefficients at each step, alpha = sqrt(sigmoid(lambda)).
    sigmas : Tensor
        Noise variances at each step, sigma^2 = 1 - alpha^2.
        
    Notes
    -----
    The linear schedule in log-SNR space corresponds to a geometric
    schedule in SNR space, providing finer discretisation at high
    noise levels where denoising is more challenging.
    
    Examples
    --------
    >>> # Create reverse diffusion with trained model
    >>> reverse_diffusion = ContinuousReverseDiffusion(
    ...     denoise_model=trained_unet,
    ...     num_steps=256
    ... )
    >>> # Generate samples
    >>> samples = reverse_diffusion(device='cuda', shape=[4, 3, 128, 128])
    """
    
    def __init__(
        self, 
        denoise_model: Callable[[Tensor, Tensor], Tensor], 
        num_steps: int, 
        lambda_min: float = -10, 
        lambda_max: float = 15,
        *args,
        **kwargs
    ) -> None:
        super().__init__()
        self.denoise_model = denoise_model
        self.num_steps = num_steps
        
        # Precompute schedule: linear in log-SNR from lambda_min to lambda_max
        self.lambdas = torch.linspace(lambda_min, lambda_max, num_steps)
        
        # "Next" values for computing transition distributions
        # At each step t, we transition from lambdas[t] to lambdas_next[t]
        self.lambdas_next = torch.cat(
            (self.lambdas[1:], torch.tensor([lambda_max], device=self.lambdas.device)), 
            dim=0
        )
        
        # Signal coefficients: alpha = sqrt(sigmoid(lambda))
        self.alphas = torch.sqrt(1 / (1 + torch.exp(-self.lambdas)))
        self.alphas_next = torch.cat(
            (self.alphas[1:], torch.tensor([0], device=self.lambdas.device)), 
            dim=0
        )
        
        # Noise variances: sigma^2 = 1 - alpha^2
        self.sigmas = 1 - self.alphas ** 2
        self.sigmas_next = torch.cat(
            (self.sigmas[1:], torch.tensor([1], device=self.lambdas.device)), 
            dim=0
        )

    @staticmethod
    def _predict_x0(
        x_noisy: Tensor, 
        sigma: Tensor, 
        alpha: Tensor, 
        epsilon_hat: Tensor
    ) -> Tensor:
        """Estimate clean sample x0 from noisy sample and predicted noise.
        
        Using the forward process equation:
            x_t = alpha * x0 + sigma * epsilon
        
        We can rearrange to estimate x0:
            x0_hat = (x_t - sigma * epsilon_hat) / alpha
        
        Parameters
        ----------
        x_noisy : Tensor
            Current noisy sample x_t.
        sigma : Tensor
            Noise standard deviation at current step.
        alpha : Tensor
            Signal coefficient at current step.
        epsilon_hat : Tensor
            Predicted noise from the denoising network.
            
        Returns
        -------
        Tensor
            Estimated clean sample x0_hat.
        """
        predicted_x0 = (x_noisy - sigma * epsilon_hat) / alpha
        return predicted_x0
    
    def reverse_mean_variance(
        self, 
        t: int, 
        x_lambda: Tensor
    ) -> tuple[Tensor, Tensor]:
        """Compute posterior mean and variance for the reverse step.
        
        Given the current noisy sample at log-SNR = lambdas[t], compute
        the mean and variance of the distribution for the sample at
        log-SNR = lambdas_next[t] (slightly less noisy).
        
        Uses Equation 3 from the classifier-free guidance paper for the
        posterior parameterisation.
        
        Parameters
        ----------
        t : int
            Current step index in the reverse process.
        x_lambda : Tensor
            Current noisy sample at step t.
            
        Returns
        -------
        mean : Tensor
            Posterior mean for x at step t+1.
        variance : Tensor
            Posterior variance for x at step t+1.
            
        Notes
        -----
        The posterior is derived from:
            delta_lambda = lambda[t] - lambda[t+1]
            c = 1 - exp(delta_lambda)  (computed as -expm1 for stability)
            
        The mean interpolates between the current sample (scaled) and
        the predicted clean sample, weighted by the log-SNR difference.
        """
        # Predict noise at current step
        lambda_t = self.lambdas[t].view(-1, 1, 1, 1).to(x_lambda.device)
        epsilon_hat = self.denoise_model(x_lambda, lambda_t)
        
        # Estimate clean image and clamp to valid range
        sigma_t = self.sigmas[t].sqrt()
        alpha_t = self.alphas[t]
        x0_hat = self._predict_x0(x_lambda, sigma_t, alpha_t, epsilon_hat)
        x0_hat = x0_hat.clamp_(-1, 1)
        
        # Compute posterior mean and variance (Eq. 3 from classifier-free guidance)
        delta_lambda = self.lambdas[t] - self.lambdas_next[t]
        
        # c = 1 - exp(delta_lambda), computed as -expm1(delta_lambda) for numerical stability
        # expm1(x) = exp(x) - 1, so -expm1(x) = 1 - exp(x)
        c = -expm1(delta_lambda)
        
        # Posterior mean: weighted combination of scaled current sample and predicted x0
        alpha_next = self.alphas_next[t]
        mean = alpha_next * (x_lambda * (1 - c) / alpha_t + c * x0_hat)
        
        # Posterior variance
        variance = self.sigmas_next[t] * c

        return mean, variance

    def reverse_sample(self, t: int, x_lambda: Tensor) -> Tensor:
        """Sample from the reverse posterior at step t.
        
        Computes the posterior mean and variance, then samples:
            x_{t+1} = mean + sqrt(variance) * z,    where z ~ N(0, I)
        
        At the final step (t = num_steps - 1), we return the mean
        without adding noise to get the final clean sample.
        
        Parameters
        ----------
        t : int
            Current step index.
        x_lambda : Tensor
            Current noisy sample.
            
        Returns
        -------
        Tensor
            Sample at the next (less noisy) step.
        """
        mean, variance = self.reverse_mean_variance(t, x_lambda)
        
        # Sample from posterior (no noise at final step)
        if t != self.num_steps - 1:
            noise = torch.randn_like(x_lambda)
            return mean + noise * variance.sqrt()
        else:
            return mean
    
    @torch.no_grad()
    def forward(
        self, 
        device: str | torch.device, 
        shape: list[int] = [1, 1, 64, 64]
    ) -> Tensor:
        """Generate samples by running the full reverse diffusion process.
        
        Starting from pure Gaussian noise, iteratively applies the reverse
        process to progressively denoise until reaching a clean sample.
        
        Parameters
        ----------
        device : str or torch.device
            Device to run generation on ('cpu', 'cuda', etc.).
        shape : list[int], default=[1, 1, 64, 64]
            Shape of samples to generate: [batch, channels, height, width].
            
        Returns
        -------
        Tensor
            Generated samples, shape as specified.
            
        Examples
        --------
        >>> reverse_diff = ContinuousReverseDiffusion(model, num_steps=128)
        >>> samples = reverse_diff(device='cuda', shape=[8, 3, 128, 128])
        >>> # samples.shape = torch.Size([8, 3, 128, 128])
        """
        print('Generating samples via reverse diffusion...')
        
        # Initialise from pure Gaussian noise
        x = torch.randn(shape, device=device)
        
        # Store intermediate samples (useful for visualisation/debugging)
        intermediates = []
        
        # Iterate through reverse process
        for step in tqdm(range(self.num_steps), desc='Reverse diffusion'):
            x = self.reverse_sample(step, x)
            intermediates.append(x.detach().cpu())
        
        # Return second-to-last sample (final sample before mean-only step)
        # This often has slightly better perceptual quality
        return intermediates[-2]

### 3.4 Training loop

For training, the data is passed to a dataloader which yields batches of data each iteration. For construction purposes, the data is first wrapped in a `TensorDataset` object, also imported from torch.

A training loop is then constructed, where samples are noised via the `ContinuousForwardDiffusion` class, the noise is predicted, the loss computed, and the parameters of the `UNet` updated. 

Below is a minimum working example of this training loop. 

In [ ]:
import itertools
import os
import torch
from torch.utils.data import DataLoader, TensorDataset
from accelerate import Accelerator
from tqdm import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt

# Configure CPU threads
torch.set_num_threads(2)
print(f"PyTorch compute threads: {torch.get_num_threads()}")

# Instantiate required objects: gpu accelerator, optimiser, UNet, diffusion, optimiser
accelerator = Accelerator()
train_ds = TensorDataset(train)
val_ds = TensorDataset(val)

train_dataloader = DataLoader(train_ds, batch_size=8)
val_dataloader = DataLoader(val_ds, batch_size=8)

loss_fn = torch.nn.MSELoss()

unet = UNet()
diffusion = ContinuousForwardDiffusion(denoise_model=unet)
optim = torch.optim.Adam(diffusion.denoise_model.parameters(), lr=3e-4, weight_decay=0.0001)

# move across to GPU, if available
train_dataloader, val_dataloader, diffusion, optim = accelerator.prepare(train_dataloader, val_dataloader, diffusion, optim)

iters = 0
train_losses = []
val_losses = []
iterations = []

for epoch in tqdm(range(100)):
    for idx, zipped_data in enumerate(zip(train_dataloader, itertools.cycle(val_dataloader))):
        # unpack training and validation data
        data, val_data = zipped_data

        # set model weights to train
        diffusion.train()

        # zero gradients of optimiser for new iteration
        optim.zero_grad(set_to_none=True)
        y = data[0] # remove leading singleton dimension
        
        # compute loss
        eps_hat, eps = diffusion(y)
        loss = loss_fn(eps_hat, eps)
        
        # clip gradients for stable training
        torch.nn.utils.clip_grad_norm_(diffusion.parameters(), max_norm=1.0)

        # backward propagation to update unet weights
        loss.backward()

        # optimiser step
        optim.step()

        # validation -- repeat above for validation data
        diffusion.eval()
        with torch.no_grad():
            y = val_data[0]
            eps_hat, eps = diffusion(y)
            val_loss = loss_fn(eps_hat, eps)
        
        # logging
        if iters % 200 == 0:
             # Store values
            iterations.append(iters)
            train_losses.append(loss.item())
            val_losses.append(val_loss.item())
            metrics_dict = {'epoch': epoch, 'iter': iters, 'train loss': loss, 'val loss': val_loss}
            print(metrics_dict)

        iters += 1

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.plot(iterations, train_losses, label='Training Loss', linewidth=2)
plt.plot(iterations, val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Training/Validation History')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()